# Inverted Pendulum: Digital Controller Design

Key Python commands used in this tutorial are: [`control.StateSpace`](https://python-control.readthedocs.io/en/latest/generated/control.StateSpace.html), [`control.c2d`](https://python-control.readthedocs.io/en/latest/generated/control.c2d.html), [`control.dlqr`](https://python-control.readthedocs.io/en/latest/generated/control.dlqr.html), [`control.forced_response`](https://python-control.readthedocs.io/en/latest/generated/control.forced_response.html)

In this digital control version of the inverted pendulum problem, we will use the state-space method to design the digital controller. If you refer to the [Inverted Pendulum: System Modeling](InvertedPendulum_SystemModeling.ipynb) page, the linearized state-space equations were derived as:

$$ \left[{\begin{array}{c}   \dot{x}\\ \ddot{x}\\ \dot{\phi}\\ \ddot{\phi} \end{array}}\right] = \left[{\begin{array}{cccc}   0&1&0&0\\   0&\frac{-(I+ml^2)b}{I(M+m)+Mml^2}&\frac{m^2gl^2}{I(M+m)+Mml^2}&0\\   0&0&0&1\\   0&\frac{-mlb}{I(M+m)+Mml^2}&\frac{mgl(M+m)}{I(M+m)+Mml^2}&0 \end{array}}\right] \left[{\begin{array}{c}   x\\ \dot{x}\\ \phi\\ \dot{\phi} \end{array}}\right]+ \left[{\begin{array}{c}0\\   \frac{I+ml^2}{I(M+m)+Mml^2}\\   0 \\   \frac{ml}{I(M+m)+Mml^2} \end{array}}\right]u $$

$${\bf y} = \left[{\begin{array}{cccc}   1&0&0&0\\0&0&1&0 \end{array}}\right] \left[{\begin{array}{c}   x\\ \dot{x}\\ \phi\\ \dot{\phi} \end{array}}\right]+ \left[{\begin{array}{c}   0\\0 \end{array}}\right]u $$

where:

``` (M)       mass of the cart                         0.5 kg (m)       mass of the pendulum                     0.2 kg (b)       coefficient of friction for cart         0.1 N/m/sec (l)       length to pendulum center of mass        0.3 m (I)       mass moment of inertia of the pendulum   0.006 kg.m^2 (F)       force applied to the cart (x)       cart position coordinate (theta)   pendulum angle from vertical (down) ```

For this problem the outputs are the cart's displacement ($x$ in meters) and the pendulum angle ($\phi$ in radians) where $\phi$ represents the deviation of the pendulum's position from equilibrium, that is, $\theta = \pi + \phi$.

The design criteria for this system for a 0.2-m step in desired cart position $x$ are as follows:

* Settling time for $x$ and $\theta$ of less than 5 seconds * Rise time for $x$ of less than 0.5 seconds * Pendulum angle $\theta$ never more than 20 degrees (0.35 radians) from the vertical * Steady-state error of less than 2% for $x$ and $\theta$

## Discrete state-space

Our first step in designing a digital controller is to convert the above continuous state-space equations to a discrete form. We will accomplish this employing the Python function `control.c2d`. This function requires that we specify three arguments: a continuous system model, the sampling time (Ts in sec/sample), and the method.

In choosing a sample time, note that it is desired that the sampling frequency be fast compared to the dynamics of the system. One measure of a system's "speed" is its closed-loop bandwidth. A good rule of thumb is that the sampling time be smaller than 1/30th of the closed-loop bandwidth frequency which can be determined from the closed-loop Bode plot.

Assuming that the closed-loop bandwidth frequencies are around 1 rad/sec for both the cart and the pendulum, let the sampling time be 1/100 sec/sample. The discretization method we will use is the zero-order hold ('zoh'). For further details, refer to the [Introduction: Digital Controller Design](../Introduction/Introduction_ControlDigital.ipynb) page. Now we are ready to use the `control.c2d` function. Enter the following commands into a code cell. Running this code will give you the discrete time state-space model.



In [ ]:
import control
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set(
    rc={
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "figure.figsize": (4 * 1.618, 4),
        "figure.dpi": 200,
    }
)

M = 0.5
m = 0.2
b = 0.1
I = 0.006
g = 9.8
l = 0.3

p = I * (M + m) + M * m * l**2  # denominator for the A and B matrices

A = np.array(
    [
        [0, 1, 0, 0],
        [0, -(I + m * l**2) * b / p, (m**2 * g * l**2) / p, 0],
        [0, 0, 0, 1],
        [0, -(m * l * b) / p, m * g * l * (M + m) / p, 0],
    ]
)
B = np.array([[0], [(I + m * l**2) / p], [0], [m * l / p]])
C = np.array([[1, 0, 0, 0], [0, 0, 1, 0]])
D = np.array([[0], [0]])

sys_ss = control.StateSpace(A, B, C, D)

Ts = 1 / 100  # Sampling time
sys_d = control.c2d(sys_ss, Ts, method="zoh")

print("Discrete system:")
print(sys_d)

Now we have obtained the discrete state-space model of the form:

$$x(k+1) = A_d x(k) + B_d u(k)$$

$$y(k) = C_d x(k) + D_d u(k)$$

## Digital controller design

Now we will design a digital controller using the discrete linear quadratic regulator (DLQR) method. This is similar to the continuous LQR design but applied to the discrete system. Let's check controllability and then design the controller:



In [ ]:
# Extract discrete matrices
Ad = sys_d.A
Bd = sys_d.B
Cd = sys_d.C
Dd = sys_d.D

# Check controllability
Co = control.ctrb(Ad, Bd)
print("Controllability matrix rank:", np.linalg.matrix_rank(Co))
print("System order:", Ad.shape[0])
print("System is controllable:", np.linalg.matrix_rank(Co) == Ad.shape[0])

# Design DLQR controller
Q = Cd.T @ Cd  # Weight matrix for states
R = np.array([[1]])  # Weight matrix for input

K, S, E = control.dlqr(Ad, Bd, Q, R)
print("\nDLQR gain matrix K:")
print(K)

Now let's examine the closed-loop response. We need to add a reference input scaling factor to eliminate steady-state error:



In [ ]:
# Closed-loop system
Ac = Ad - Bd @ K
Bc = Bd
Cc = Cd
Dc = Dd

sys_cl = control.StateSpace(Ac, Bc, Cc, Dc, dt=Ts)

# Reference input scaling
# For a step input, we need to compute Nbar to eliminate steady-state error
# Nbar = 1 / (C * inv(I - A + B*K) * B)
I_mat = np.eye(Ac.shape[0])
Nbar = 1 / (Cd @ np.linalg.inv(I_mat - Ac) @ Bd)
Nbar = Nbar[0, 0]  # Extract scalar value

# Closed-loop system with reference input
sys_cl_ref = control.StateSpace(Ac, Bc * Nbar, Cc, Dc, dt=Ts)

# Step response
t = np.arange(0, 5, Ts)
r = 0.2 * np.ones_like(t)  # 0.2 m step command
T, yout, _ = control.forced_response(sys_cl_ref, T=t, U=r, X0=np.zeros((4,)))

plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
plt.plot(T, yout[0, :], label="Cart position (m)")
plt.axhline(y=0.2, color="r", linestyle="--", label="Reference")
plt.xlabel("Time (s)")
plt.ylabel("Cart Position (m)")
plt.title("Step Response: Cart Position (Digital Control)")
plt.legend()
plt.grid("on")

plt.subplot(2, 1, 2)
plt.plot(T, yout[1, :], label="Pendulum angle (rad)")
plt.axhline(y=0.35, color="r", linestyle="--", label="Max angle")
plt.axhline(y=-0.35, color="r", linestyle="--")
plt.xlabel("Time (s)")
plt.ylabel("Pendulum Angle (rad)")
plt.title("Step Response: Pendulum Angle (Digital Control)")
plt.legend()
plt.grid("on")

plt.tight_layout()
plt.show()

The closed-loop poles of the discrete system can be found by examining the eigenvalues of the closed-loop system matrix:



In [ ]:
closed_loop_poles = np.linalg.eigvals(Ac)
print("Closed-loop poles:", closed_loop_poles)
print("\nAll poles are inside unit circle:", np.all(np.abs(closed_loop_poles) < 1))